
# Indian PII Masking Project

## Setup

1. Download all project files and `requirements.txt`.
2. Install dependencies:

```bash
pip install -r requirements.txt
```

3. Place your training dataset in the same folder as the project files.
4. Your dataset must contain the same columns as `pii_dataset_v3.xlsx`.

## Training

1. Open `train_gliner_pii.py`.
2. Go to approximately line 198 inside the `train()` function.
3. Change:

```python
xlsx_path="pii_dataset_3000.xlsx"
```

to the filename of your dataset.

4. Run:

```bash
python train_gliner_pii.py
```

5. Wait for training to complete. The fine-tuned model will be saved to the output directory.

## Starting the API

Run:

```bash
uvicorn api:app --reload --port 8000
```

## Running the Batch Masker

Open a second terminal and run:

```bash
python run_pii_masker.py
```

This will send the input queries to the API and generate masked outputs.


# requirements.txt


In [ ]:
accelerate==1.13.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.0
aiosignal==1.4.0
annotated-doc==0.0.4
annotated-types==0.7.0
anyio==4.13.0
attrs==26.1.0
blis==1.3.3
catalogue==2.0.10
certifi==2026.5.20
cffi==2.0.0
charset-normalizer==3.4.7
click==8.4.1
cloudpathlib==0.24.0
colorama==0.4.6
confection==1.3.3
cryptography==48.0.0
curated-tokenizers==0.0.9
curated-transformers==0.1.1
cymem==2.0.13
datasets==5.0.0
dill==0.4.1
en_core_web_lg @ https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl#sha256=293e9547a655b25499198ab15a525b05b9407a75f10255e405e8c3854329ab63
en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl#sha256=1932429db727d4bff3deed6b34cfc05df17794f4a52eeb26cf8928f7c1a0fb85
en_core_web_trf @ https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.8.0/en_core_web_trf-3.8.0-py3-none-any.whl#sha256=272a31e9d8530d1e075351d30a462d7e80e31da23574f1b274e200f3fff35bf5
et_xmlfile==2.0.0
fastapi==0.136.3
filelock==3.29.0
flatbuffers==25.12.19
frozenlist==1.8.0
fsspec==2026.4.0
ftfy==6.3.1
gliner==0.2.26
h11==0.16.0
hf-xet==1.5.0
httpcore==1.0.9
httpx==0.28.1
huggingface_hub==1.16.4
idna==3.16
iniconfig==2.3.0
jellyfish==1.2.1
Jinja2==3.1.6
langcodes==3.5.1
langdetect==1.0.9
locate==1.1.1
markdown-it-py==4.2.0
MarkupSafe==3.0.3
mdurl==0.1.2
mpmath==1.3.0
msgpack==1.1.2
multidict==6.7.1
multiprocess==0.70.19
murmurhash==1.0.15
networkx==3.6.1
numpy==2.4.4
onnxruntime==1.26.0
openpyxl==3.1.5
packaging==26.0
pandas==3.0.3
phonenumbers==9.0.31
pillow==12.2.0
pluggy==1.6.0
preshed==3.0.13
presidio_analyzer==2.2.362
presidio_anonymizer==2.2.362
propcache==0.5.2
protobuf==7.35.0
psutil==7.2.2
pyarrow==24.0.0
pycparser==3.0
pydantic==2.13.4
pydantic_core==2.46.4
Pygments==2.20.0
pytest==9.0.3
pytest-asyncio==1.4.0
python-dateutil==2.9.0.post0
python-whois==0.9.6
PyYAML==6.0.3
regex==2026.5.9
requests==2.34.2
requests-file==3.0.1
rich==15.0.0
safetensors==0.7.0
sentencepiece==0.2.1
shellingham==1.5.4
six==1.17.0
smart_open==7.6.1
spacy==3.8.14
spacy-curated-transformers==0.3.1
spacy-legacy==3.0.12
spacy-loggers==1.0.5
srsly==2.5.3
starlette==1.2.0
sympy==1.14.0
thinc==8.3.13
tldextract==5.3.1
tokenizers==0.22.2
torch==2.12.0+cu130
torchaudio==2.11.0+cu130
torchvision==0.27.0+cu130
tqdm==4.67.3
transformers==5.1.0
typer==0.25.1
typer-slim==0.24.0
typing-inspection==0.4.2
typing_extensions==4.15.0
tzdata==2026.2
urllib3==2.7.0
uvicorn==0.48.0
wasabi==1.1.3
wcwidth==0.8.1
weasel==1.0.0
wordfreq==3.1.1
wrapt==2.2.1
xxhash==3.7.0
yarl==1.24.2


# train_gliner_pii.py


In [ ]:
"""
Fine-tune GLiNER on Indian PII dataset (pii_dataset_v3.xlsx)
Compatible with gliner 0.2.26

Fixes applied vs previous versions:
  - No GLiNERDataset (removed in 0.2.x) → ListDataset + SpanDataCollator
  - tokenizer= → processing_class= (HF Transformers ≥ 4.46)
  - SafeCollator: injects ALL_LABELS as fallback negatives when a batch
    contains only no-pii/noise rows (ner=[]) → prevents reshape([8,-1,0]) crash
  - fp16=True when CUDA is available, False otherwise (no manual --device needed)
"""

import re
import os
import json
import random
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from typing import Any

# ── 1. Entity label normalisation ────────────────────────────────────────────
LABEL_MAP = {
    "full_name":       "full_name",
    "name":            "full_name",
    "phone_number":    "phone_number",
    "aadhaar_number":  "aadhaar_number",
    "pan_card":        "pan_card",
    "company_name":    "company_name",
    "postal_address":  "postal_address",
    "date_of_birth":   "date_of_birth",
    "bank_account":    "bank_account",
    "email_address":   "email_address",
    "passport_number": "passport_number",
    "voter_id":        "voter_id",
    "drivers_license": "drivers_license",
    "card_number":     "card_number",
}

ALL_LABELS = sorted(set(LABEL_MAP.values()))


def normalise_label(raw: str) -> str | None:
    return LABEL_MAP.get(raw.strip().lower())


# ── 2. (Query, Masked) → GLiNER sample ───────────────────────────────────────

def build_sample(query: str, masked: str) -> dict | None:
    if not isinstance(query, str) or not isinstance(masked, str):
        return None
    query  = query.strip()
    masked = masked.strip()

    placeholder_re = re.compile(r'\[([A-Za-z_]+)\]')
    placeholders   = list(placeholder_re.finditer(masked))

    tokens = query.split()
    if not tokens:
        return None

    token_starts, token_ends = [], []
    pos = 0
    for tok in tokens:
        idx = query.index(tok, pos)
        token_starts.append(idx)
        token_ends.append(idx + len(tok))
        pos = idx + len(tok)

    def char_to_tok(char_idx, side="start"):
        for i, (ts, te) in enumerate(zip(token_starts, token_ends)):
            if side == "start" and ts <= char_idx < te:
                return i
            if side == "end"   and ts < char_idx <= te:
                return i
        dists = [abs(ts - char_idx) for ts in token_starts]
        return dists.index(min(dists))

    if not placeholders:
        return {"tokenized_text": tokens, "ner": []}

    ner_spans = []
    q_pos, m_pos = 0, 0

    for i, ph in enumerate(placeholders):
        label = normalise_label(ph.group(1))
        if label is None:
            return None

        prefix_len = ph.start() - m_pos
        if prefix_len < 0:
            return None
        q_pos += prefix_len
        m_pos  = ph.end()

        next_ph    = placeholders[i + 1] if i + 1 < len(placeholders) else None
        after_text = masked[m_pos:next_ph.start()] if next_ph else masked[m_pos:]

        if after_text:
            idx = query.find(after_text, q_pos)
            if idx == -1:
                idx = query.find(after_text.strip(), q_pos)
                if idx == -1:
                    return None
            span_end = idx
        else:
            span_end = len(query)

        if not query[q_pos:span_end].strip():
            return None

        ner_spans.append((q_pos, span_end, label))
        q_pos = span_end

    ner_token_spans = []
    for cs, ce, lbl in ner_spans:
        t_start = char_to_tok(cs, "start")
        t_end   = char_to_tok(ce, "end")
        if t_start <= t_end:
            ner_token_spans.append([t_start, t_end, lbl])

    return {"tokenized_text": tokens, "ner": ner_token_spans}


# ── 3. Load dataset ───────────────────────────────────────────────────────────

def load_dataset(xlsx_path: str, max_rows: int | None = None) -> list[dict]:
    df = pd.read_excel(xlsx_path, nrows=max_rows)
    df = df[["Query", "Masked"]].dropna(subset=["Query", "Masked"])

    samples, skipped = [], 0
    for _, row in df.iterrows():
        s = build_sample(str(row["Query"]), str(row["Masked"]))
        if s is not None:
            samples.append(s)
        else:
            skipped += 1

    print(f"Loaded {len(samples)} samples  |  {skipped} skipped (alignment errors)")
    return samples


# ── 4. PyTorch Dataset wrapper ────────────────────────────────────────────────

class ListDataset(Dataset):
    def __init__(self, data: list[dict]):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]


# ── 5. Safe collator ──────────────────────────────────────────────────────────
# Root cause of reshape([8, -1, 0]):
#   When every sample in a batch has ner=[], batch_generate_class_mappings
#   builds an empty types list → class_to_id={} → num_classes=0.
#   The model then tries reshape(batch, seq, 0) which crashes.
#
# Fix: subclass SpanDataCollator and inject ALL_LABELS as ner_negatives on
# every sample so there's always at least one class in the batch.

class SafeCollator:
    """
    Wraps SpanDataCollator. Before collating, stamps ner_negatives=ALL_LABELS
    on every sample so batches of pure negatives (no-pii/noise rows) never
    produce an empty class set.
    """
    def __init__(self, base_collator, all_labels: list[str]):
        self.base      = base_collator
        self.all_labels = all_labels

    def __call__(self, batch: list[dict[str, Any]]) -> dict[str, Any]:
        # Inject fallback negatives so num_classes is always ≥ 1
        patched = []
        for sample in batch:
            s = dict(sample)
            if not s.get("ner"):
                s["ner_negatives"] = self.all_labels
            patched.append(s)
        return self.base(patched)


# ── 6. Train/val split ────────────────────────────────────────────────────────

def split(samples, val_ratio=0.1, seed=42):
    random.seed(seed)
    data = samples.copy()
    random.shuffle(data)
    n_val = max(1, int(len(data) * val_ratio))
    return data[n_val:], data[:n_val]


# ── 7. Fine-tune ──────────────────────────────────────────────────────────────

def train(
    xlsx_path:   str        = "pii_dataset_3000.xlsx",
    base_model:  str        = "urchade/gliner_medium-v2.1",
    output_dir:  str        = "./gliner_pii_finetuned",
    num_epochs:  int        = 5,
    batch_size:  int        = 8,
    lr:          float      = 5e-5,
    val_ratio:   float      = 0.1,
    max_rows:    int | None = None,   # None = use full dataset
):
    from gliner import GLiNER
    from gliner.training import Trainer, TrainingArguments
    from gliner.data_processing.collator import SpanDataCollator

    use_cuda = torch.cuda.is_available()
    print(f"CUDA available: {use_cuda}" + (f"  →  {torch.cuda.get_device_name(0)}" if use_cuda else "  →  training on CPU"))

    print(f"Loading base model: {base_model}")
    model = GLiNER.from_pretrained(base_model)

    base_collator = SpanDataCollator(
        config         = model.config,
        data_processor = model.data_processor,
        prepare_labels = True,
    )
    collator = SafeCollator(base_collator, ALL_LABELS)

    print("Parsing dataset …")
    samples = load_dataset(xlsx_path, max_rows=max_rows)
    train_data, val_data = split(samples, val_ratio=val_ratio)
    print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

    Path("gliner_train_data.json").write_text(json.dumps(train_data[:20], indent=2))
    print("Saved gliner_train_data.json (first 20 samples for inspection)")

    training_args = TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = num_epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = lr,
        weight_decay                = 0.01,
        warmup_ratio                = 0.1,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        logging_steps               = 50,
        fp16                        = use_cuda,   # auto: True on GPU, False on CPU
        dataloader_num_workers      = 0,
        focal_loss_alpha            = -1,
        focal_loss_gamma            = 0,
        negatives                   = 1.0,
        masking                     = "global",
    )

    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = ListDataset(train_data),
        eval_dataset     = ListDataset(val_data),
        data_collator    = collator,
        processing_class = model.data_processor.transformer_tokenizer,
    )

    print("Training …")
    trainer.train()

    print(f"Saving to {output_dir}")
    model.save_pretrained(output_dir)
    print("Done.")


# ── 8. Quick inference test ───────────────────────────────────────────────────

def test(model_dir: str = "./gliner_pii_finetuned"):
    from gliner import GLiNER

    model = GLiNER.from_pretrained(model_dir)
    cases = [
        "My aadhaar is 1234 5678 9012 and pan is ABCDE1234F",
        "Contact me at ravi.sharma@gmail.com or 9876543210",
        "I Priya Verma am the proprietor of ABC Enterprises",
        "aadhaar 5868 2635 4341 pan GXZEN9431O phone 7947604856",
        "I wish to bring to your kind notice that the order is issued long back.",
    ]
    for text in cases:
        entities = model.predict_entities(text, ALL_LABELS, threshold=0.5)
        print(f"\nText : {text}")
        print(f"Found: {[(e['text'], e['label']) for e in entities]}")


# ── 9. CLI ────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import argparse

    p = argparse.ArgumentParser(description="Fine-tune GLiNER on Indian PII data")
    p.add_argument("--xlsx",        default="pii_dataset_v3.xlsx")
    p.add_argument("--base_model",  default="urchade/gliner_medium-v2.1")
    p.add_argument("--output_dir",  default="./gliner_pii_finetuned")
    p.add_argument("--epochs",      type=int,   default=5)
    p.add_argument("--batch_size",  type=int,   default=8)
    p.add_argument("--lr",          type=float, default=5e-5)
    p.add_argument("--max_rows",    type=int,   default=None,
                   help="Cap rows loaded from xlsx (e.g. 200 for a quick test run)")
    p.add_argument("--test_only",   action="store_true")
    args = p.parse_args()

    if args.test_only:
        test(args.output_dir)
    else:
        train(
            xlsx_path  = args.xlsx,
            base_model = args.base_model,
            output_dir = args.output_dir,
            num_epochs = args.epochs,
            batch_size = args.batch_size,
            lr         = args.lr,
            max_rows   = args.max_rows,
        )
        test(args.output_dir)

# indian_pii_masker(2).py


In [ ]:
"""
Indian PII Masker
=================
Entities masked
───────────────────────────────────────────────────────────
  AADHAAR        12-digit UID  (4-4-4, any separator)
  PAN            Permanent Account Number  (AAAAA9999A)
  GST            GSTIN  (29ABCDE1234F1Z5)
  IFSC           Bank IFSC code  (HDFC0001234)
  PHONE          Indian mobile numbers (all common formats)
  ACCOUNT        Bank account numbers (context-anchored, 11-18 digits)
  VOTER_ID       EPIC number  (3 letters + 7 digits)
  PASSPORT       Indian passport  (1 letter + 7 digits)
  DL             Driving licence  (SS-RR-YYYY-NNNNNNN)
#   UDYAM          MSME Udyam registration  (UDYAM-XX-DD-NNNNNNN)
  UAN            EPFO Universal Account Number (context-anchored, 12 digits)
  EMAIL          E-mail addresses

Ambiguous formats supported (NEW)
───────────────────────────────────────────────────────────
  All entities now tolerate arbitrary separators (spaces, dashes, dots,
  underscores, slashes) inserted between ANY characters, e.g.:
    E C P P G 0 1 1 1 K       ← space after every character
    E-C-P-P-G-0-1-1-1-K       ← dash after every character
    ECPP G0 111K               ← random groupings
    [Aadhaar Redacted]         ← dashes instead of spaces in Aadhaar
    98765 43210                ← phone split randomly
    MH-27-2012-0034761         ← DL with extra dashes

  A pre-processing step (_normalize_dense_separators) collapses
  sequences where every character is followed by a separator — the
  dominant "manual obfuscation" pattern — into compact tokens before
  the main analysis runs.

Setup
─────
    pip install presidio-analyzer presidio-anonymizer spacy psutil
    python -m spacy download en_core_web_lg
"""

import re
import time
import os
import psutil

from presidio_analyzer import (
    AnalyzerEngine,
    PatternRecognizer,
    Pattern,
    RecognizerResult,
)
from presidio_analyzer.entity_recognizer import EntityRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig


# =========================================================
# SEPARATOR CONSTANT — inserted between every character/group
# in "ambiguous" regex patterns
# =========================================================

# Matches 0-to-many of: space, tab, dash, dot, underscore, slash, pipe
_S = r'[\s.\-_/|]{0,3}'

# =========================================================
# HELPERS
# =========================================================

def _norm(text: str) -> str:
    """Strip everything except letters and digits (case-preserved)."""
    return re.sub(r'[^A-Za-z0-9]', '', text)


def _sep_pattern(strict: str) -> str:
    """
    Given a strict regex like r'[A-Z]{4}0[A-Z0-9]{6}', inject _S between
    every atom so it tolerates arbitrary separators between characters.

    Handles character classes [...], quantified groups {n}, and bare chars.
    Each top-level atom is kept; _S is inserted between consecutive atoms.

    This is a best-effort approach sufficient for fixed-length ID patterns.
    For more complex patterns, recognizers build their own explicitly.
    """
    # Tokenise into atoms: [...], {n}, bare letter/digit/escaped, ^/$
    token_re = re.compile(
        r'(\[\^?[^\]]*\]\{?\d*,?\d*\}?'   # [class]{n}
        r'|\[\^?[^\]]*\]'                 # [class]
        r'|\([^)]*\)\{?\d*,?\d*\}?'        # (group){n}
        r'|\{?\d+,?\d*\}'                  # bare {n}
        r'|\\.'                              # escaped char
        r'|[^^$.|?*+(){}\\]'                # bare char (not meta)
        r'|[.^$|?*+(){}\\]'                 # meta
        r')'
    )
    tokens = token_re.findall(strict)
    return _S.join(tokens)


# =========================================================
# PRE-PROCESSOR — collapse "space/dash after every char" patterns
# =========================================================

def _normalize_dense_separators(text: str) -> str:
    """
    Collapse sequences where nearly every alphanumeric character is
    separated by a consistent single separator, e.g.:

        "E C P P G 0 1 1 1 K"  →  "ECPPG0111K"
        "E-C-P-P-G-0-1-1-1-K"  →  "ECPPG0111K"
        "2 3 4 5  6 7 8 9  0 1 2 3"  →  "2345678901 23"  (only tight runs)

    Algorithm:
      - Find maximal runs of (single-alnum)(single-separator) followed
        by a final alnum — at least 4 characters long.
      - Replace each run with its stripped version.
      - Preserve surrounding context so span offsets remain consistent
        for the result text (positions shift, but that is fine because
        Presidio operates on the normalised copy).
    """
    # Pattern: alnum, then (sep, alnum) repeated ≥3 times
    # sep = exactly one non-alnum non-newline char (space, dash, dot, underscore…)
    dense = re.compile(
        r'(?<![A-Za-z0-9])'        # not preceded by alnum (word boundary)
        r'([A-Za-z0-9]'            # first char
        r'(?:[^A-Za-z0-9\n][A-Za-z0-9]){3,})'  # (sep + alnum) × 3+
        r'(?![A-Za-z0-9])'         # not followed by alnum
    )

    def _strip(m):
        return re.sub(r'[^A-Za-z0-9]', '', m.group(0))

    return dense.sub(_strip, text)


# =========================================================
# VALIDATORS  (unchanged — all use _norm internally)
# =========================================================

def _valid_aadhaar(text: str) -> bool:
    t = _norm(text)
    return len(t) == 12 and t.isdigit() and t[0] not in "01"


def _valid_pan(text: str) -> bool:
    return bool(re.fullmatch(r'[A-Z]{5}[0-9]{4}[A-Z]', _norm(text).upper()))


def _valid_ifsc(text: str) -> bool:
    return bool(re.fullmatch(r'[A-Z]{4}0[A-Z0-9]{6}', _norm(text).upper()))


def _valid_gst(text: str) -> bool:
    return bool(re.fullmatch(
        r'\d{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]',
        _norm(text).upper(),
    ))


def _valid_phone(text: str) -> bool:
    t = _norm(text)
    if   t.startswith("91")  and len(t) == 12: t = t[2:]
    elif t.startswith("091") and len(t) == 13: t = t[3:]
    elif t.startswith("0")   and len(t) == 11: t = t[1:]
    return len(t) == 10 and t.isdigit() and t[0] in "6789"


def _valid_account(text: str) -> bool:
    digits = re.search(r'\d{11,18}', _norm(text))
    if not digits:
        return False
    t = digits.group()
    return t.isdigit() and 11 <= len(t) <= 18


def _valid_voter_id(text: str) -> bool:
    return bool(re.fullmatch(r'[A-Z]{3}[0-9]{7}', _norm(text).upper()))


def _valid_passport(text: str) -> bool:
    t = _norm(text).upper()
    return bool(re.fullmatch(r'[A-Z][1-9]\d{5}[1-9]', t))


def _valid_dl(text: str) -> bool:
    t = _norm(text).upper()
    return bool(re.fullmatch(r'[A-Z]{2}\d{2}(19|20)\d{2}\d{7}', t))


# def _valid_udyam(text: str) -> bool:
#     return bool(re.fullmatch(
#         r'UDYAM[A-Z]{2}\d{2}\d{7}',
#         _norm(text).upper(),
#     ))


def _valid_uan(text: str) -> bool:
    digits = re.search(r'\d{12}', _norm(text))
    return bool(digits)


# =========================================================
# RECOGNIZERS
# =========================================================

class CustomEmailRecognizer(PatternRecognizer):
    """
    Explicit email recognizer with strict fallbacks for typos and obfuscation.
    Tightened to prevent false positives on regular English text.
    """
    def __init__(self):
        super().__init__(
            supported_entity="EMAIL_ADDRESS",
            patterns=[
                # 1. Standard well-formed email
                Pattern(
                    "email_standard", 
                    r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b', 
                    score=1.0
                ),
                
                # 2. Missing '@' but attached directly to a known provider (e.g., ramesh1972gmail.com)
                Pattern(
                    "email_missing_at", 
                    r'(?i)\b[A-Z0-9._%+-]+(?:gmail|yahoo|outlook|hotmail|rediffmail)\.com\b', 
                    score=0.85
                ),
                
                # 3. Explicit bracket obfuscation (e.g., user[at]domain[dot]com)
                Pattern(
                    "email_obfuscated_brackets", 
                    r'(?i)\b[A-Z0-9._%+-]+\s*(?:\[at\]|\(at\))\s*[A-Z0-9.-]+\s*(?:\[dot\]|\(dot\)|\.)\s*(?:com|in|co\.in|org|net)\b', 
                    score=0.80
                ),

                # 4. Spelled out words (e.g., user at gmail dot com)
                Pattern(
                    "email_obfuscated_words", 
                    r'(?i)\b[A-Z0-9._%+-]+\s+at\s+(?:gmail|yahoo|outlook|hotmail|rediffmail)\s+(?:dot|\.)\s+(?:com|in|co\.in|org|net)\b', 
                    score=0.80
                ),
                
                # 5. Missing '@' replaced by a space (e.g., jayjagannath5press gmail.com)
                # {3,} requires the username to be 3+ chars to avoid masking "to gmail.com"
                Pattern(
                    "email_space_missing_at",
                    r'(?i)\b[A-Z0-9._%+-]{3,}\s+(?:gmail|yahoo|outlook|hotmail|rediffmail)\.com\b',
                    score=0.80
                )
            ],
        )

class AadhaarRecognizer(PatternRecognizer):
    """
    Matches Aadhaar in all separator variants:
      • 4-4-4 with standard separators  (original)
      • 4-4-4 with arbitrary separators per group
      • Every digit separated individually: [Aadhaar Redacted]
      • Mixed: [Aadhaar Redacted]
    """
    _G4  = r'\d' + _S + r'\d' + _S + r'\d' + _S + r'\d'   # 4 digits with seps
    _SEP = r'[\s.\-_()/]{0,3}'

    def __init__(self):
        g4 = self._G4
        sep = self._SEP
        super().__init__(
            supported_entity="AADHAAR",
            patterns=[
                # Original: 4-4-4 grouped
                Pattern("aadhaar_4_4_4",
                        r'(?<!\d)\d{4}' + sep + r'\d{4}' + sep + r'\d{4}(?!\d)',
                        score=0.85),
                # Fully separated: every digit has a separator
                Pattern("aadhaar_separated",
                        r'(?<![A-Za-z0-9])' + g4 + sep + g4 + sep + g4 + r'(?![A-Za-z0-9])',
                        score=0.80),
                # Bare 12 digits (context-anchored)
                Pattern("aadhaar_12_bare",
                        r'(?<!\d)\d{12}(?!\d)',
                        score=0.55),
            ],
            context=["aadhaar", "aadhar", "uid", "unique identification"],
        )

    def validate_result(self, pattern_text):
        return _valid_aadhaar(pattern_text)


class PANRecognizer(PatternRecognizer):
    """
    PAN card: AAAAA9999A
    Tolerates any separator between every character.
    Examples:
      ECPPG0111K  /  E C P P G 0 1 1 1 K  /  E-C-P-P-G-0-1-1-1-K
      ECPP-G01-11K  /  E.C.P.P.G.0.1.1.1.K
    """
    # 5 alpha, 4 digit, 1 alpha — each char separated by _S
    _ALPHA = r'[A-Za-z]'
    _DIGIT = r'[0-9]'

    def __init__(self):
        a, d, s = self._ALPHA, self._DIGIT, _S
        pan_sep = (
            r'(?<![A-Za-z0-9])'
            + a + s + a + s + a + s + a + s + a  # 5 alpha
            + s
            + d + s + d + s + d + s + d           # 4 digit
            + s
            + a                                    # 1 alpha
            + r'(?![A-Za-z0-9])'
        )
        super().__init__(
            supported_entity="PAN",
            patterns=[
                Pattern("pan_strict",
                        r'(?<![A-Z0-9])[A-Z]{5}[0-9]{4}[A-Z](?![A-Z0-9])',
                        score=0.90),
                Pattern("pan_separated",
                        pan_sep,
                        score=0.85),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_pan(pattern_text)


class GSTRecognizer(PatternRecognizer):
    """
    GSTIN: 2-digit state + PAN(10) + 1 + Z + 1 = 15 chars
    Tolerates separators between every character.
    """
    _D = r'[0-9]'
    _A = r'[A-Za-z]'
    _AN = r'[A-Za-z0-9]'

    def __init__(self):
        d, a, an, s = self._D, self._A, self._AN, _S

        gst_sep = (
            r'(?<![A-Za-z0-9])'
            # 2-digit state code
            + d + s + d + s
            # 5 alpha (PAN part)
            + a + s + a + s + a + s + a + s + a + s
            # 4 digit
            + d + s + d + s + d + s + d + s
            # 1 alpha
            + a + s
            # 1 alphanumeric (1-9 or A-Z)
            + an + s
            # literal Z
            + r'[Zz]' + s
            # 1 alphanumeric checksum
            + an
            + r'(?![A-Za-z0-9])'
        )
        super().__init__(
            supported_entity="GST",
            patterns=[
                Pattern("gst_strict",
                        r'(?<!\w)\d{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z](?!\w)',
                        score=0.95),
                Pattern("gst_separated",
                        gst_sep,
                        score=0.88),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_gst(pattern_text)


class IFSCRecognizer(PatternRecognizer):
    """
    IFSC: 4 alpha + 0 + 6 alphanumeric
    Tolerates separators between every character.
    Examples:
      HDFC0001234  /  H D F C 0 0 0 1 2 3 4  /  HDFC-0-001234
    """
    _A = r'[A-Za-z]'
    _AN = r'[A-Za-z0-9]'

    def __init__(self):
        a, an, s = self._A, self._AN, _S
        ifsc_sep = (
            r'(?<![A-Za-z0-9])'
            + a + s + a + s + a + s + a   # 4 alpha
            + s + r'0' + s                # literal 0
            + an + s + an + s + an + s + an + s + an + s + an  # 6 alphanum
            + r'(?![A-Za-z0-9])'
        )
        super().__init__(
            supported_entity="IFSC",
            patterns=[
                Pattern("ifsc_strict",
                        r'(?<![A-Z0-9])[A-Z]{4}0[A-Z0-9]{6}(?![A-Z0-9])',
                        score=0.90),
                Pattern("ifsc_separated",
                        ifsc_sep,
                        score=0.85),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_ifsc(pattern_text)


class IndianPhoneRecognizer(PatternRecognizer):
    """
    All common Indian mobile formats + fully separated digit runs.
    Examples:
      9876543210  /  98765 43210  /  9 8 7 6 5 4 3 2 1 0
      +91-98765-43210  /  +91 9 8 7 6 5 4 3 2 1 0
    """
    _CC  = r'(?:(?:\+|0{0,2})91[\s()\-]*)?'
    _S10 = (                                                 # 10 digits, any sep
        r'[6-9]' + _S
        + r'\d' + _S + r'\d' + _S + r'\d' + _S + r'\d'   # 5 digits
        + _S
        + r'\d' + _S + r'\d' + _S + r'\d' + _S + r'\d' + _S + r'\d'  # 5 digits
    )
    _P1  = r'(?<!\d)' + _CC + r'[6-9]\d{4}[\s\-.]?\d{5}(?!\d)'
    _P2  = r'(?:(?:\+|0{0,2})91[\s]*)\(\d{3}\)[\s\-]*\d{3}[\s\-]*\d{4}'
    _P3  = r'(?<!\d)0\d{2}[\s\-.]?\d{3}[\s\-.]?\d{5}(?!\d)'
    _P4  = r'(?<!\d)[6-9]\d{2}[\s\-]\d{3}[\s\-]\d{4}(?!\d)'

    def __init__(self):
        super().__init__(
            supported_entity="PHONE",
            patterns=[
                Pattern("phone_p1", self._P1, score=0.85),
                Pattern("phone_p2", self._P2, score=0.85),
                Pattern("phone_p3", self._P3, score=0.80),
                Pattern("phone_p4", self._P4, score=0.75),
                # Fully separated 10-digit run (with optional country code)
                Pattern("phone_sep10",
                        r'(?<![A-Za-z0-9])' + self._CC + self._S10 + r'(?![A-Za-z0-9])',
                        score=0.78),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_phone(pattern_text)


class VoterIdRecognizer(PatternRecognizer):
    """
    EPIC: 3 uppercase letters + 7 digits
    Tolerates separators between every character.
    Examples:
      XGN3002623  /  X G N 3 0 0 2 6 2 3  /  XGN-3002623  /  X-G-N-3-0-0-2-6-2-3
    """
    _A = r'[A-Za-z]'
    _D = r'[0-9]'

    def __init__(self):
        a, d, s = self._A, self._D, _S
        voter_sep = (
            r'(?<![A-Za-z0-9])'
            + a + s + a + s + a   # 3 alpha
            + s
            + d + s + d + s + d + s + d + s + d + s + d + s + d  # 7 digits
            + r'(?![A-Za-z0-9])'
        )
        super().__init__(
            supported_entity="VOTER_ID",
            patterns=[
                Pattern("voter_id_strict",
                        r'(?<![A-Z0-9])[A-Z]{3}[0-9]{7}(?![A-Z0-9])',
                        score=0.85),
                Pattern("voter_id_separated",
                        voter_sep,
                        score=0.80),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_voter_id(pattern_text)


class PassportRecognizer(PatternRecognizer):
    """
    Indian passport: 1 letter + 7 digits (first & last digit non-zero)
    Tolerates separators between every character.
    Examples:
      A2345671  /  A 2 3 4 5 6 7 1  /  A-2345671  /  A-2-3-4-5-6-7-1
    """
    _A = r'[A-Za-z]'
    _NZ = r'[1-9]'   # non-zero digit
    _D  = r'[0-9]'

    def __init__(self):
        a, nz, d, s = self._A, self._NZ, self._D, _S
        pp_sep = (
            r'(?<![A-Za-z0-9])'
            + a + s + nz + s          # letter + non-zero
            + d + s + d + s + d + s + d + s + d  # 5 middle digits
            + s + nz                   # non-zero last
            + r'(?![A-Za-z0-9])'
        )
        super().__init__(
            supported_entity="PASSPORT",
            patterns=[
                Pattern("passport_strict",
                        r'(?<![A-Z0-9])[A-Z][1-9]\d{5}[1-9](?![A-Z0-9])',
                        score=0.85),
                Pattern("passport_separated",
                        pp_sep,
                        score=0.80),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_passport(pattern_text)


class DrivingLicenceRecognizer(PatternRecognizer):
    """
    Indian DL: SS-RR-YYYY-NNNNNNN (2+2+4+7 = 15 chars)
    Original pattern already tolerates some separators; now also matches
    fully character-separated variants.
    Examples:
      MH272012 0034761  /  MH-27-2012-0034761  /  M H 2 7 2 0 1 2 0 0 3 4 7 6 1
    """
    _DL_STRICT = (
        r'(?<![A-Z0-9])'
        r'[A-Z]{2}[\s\-]?\d{2}[\s\-]?(19|20)\d{2}[\s\-]?\d{7}'
        r'(?!\d)'
    )
    _A = r'[A-Za-z]'
    _D = r'[0-9]'

    def __init__(self):
        a, d, s = self._A, self._D, _S
        dl_sep = (
            r'(?<![A-Za-z0-9])'
            + a + s + a                                       # 2-char state
            + s
            + d + s + d                                       # 2-digit RTO
            + s
            + r'(?:19|20)' + s + d + s + d           # year
            + s
            + d + s + d + s + d + s + d + s + d + s + d + s + d  # 7-digit serial
            + r'(?![A-Za-z0-9])'
        )
        super().__init__(
            supported_entity="DL",
            patterns=[
                Pattern("dl_strict", self._DL_STRICT, score=0.85),
                Pattern("dl_separated", dl_sep, score=0.80),
            ],
        )

    def validate_result(self, pattern_text):
        return _valid_dl(pattern_text)


# class UdyamRecognizer(PatternRecognizer):
#     """
#     UDYAM-XX-DD-NNNNNNN
#     The UDYAM prefix is distinctive; also tolerates separators within each segment.
#     Examples:
#       UDYAM-DL-04-0012345  /  U D Y A M - D L - 0 4 - 0 0 1 2 3 4 5
#     """
#     def __init__(self):
#         s = _S
#         udyam_sep = (
#             r'(?i)(?<!\w)'
#             r'U' + s + r'D' + s + r'Y' + s + r'A' + s + r'M'
#             + s + r'[\-]?' + s
#             + r'[A-Za-z]' + s + r'[A-Za-z]'
#             + s + r'[\-]?' + s
#             + r'[0-9]' + s + r'[0-9]'
#             + s + r'[\-]?' + s
#             + r'[0-9]' + s + r'[0-9]' + s + r'[0-9]' + s
#             + r'[0-9]' + s + r'[0-9]' + s + r'[0-9]' + s + r'[0-9]'
#             + r'(?!\w)'
#         )
#         super().__init__(
#             supported_entity="UDYAM",
#             patterns=[
#                 Pattern("udyam_strict",
#                         r'(?i)(?<!\w)UDYAM[\-][A-Z]{2}[\-]\d{2}[\-]\d{7}(?!\w)',
#                         score=0.99),
#                 Pattern("udyam_separated",
#                         udyam_sep,
#                         score=0.92),
#             ],
#         )
#
#     def validate_result(self, pattern_text):
#         return _valid_udyam(pattern_text)


# =========================================================
# CONTEXT-ANCHORED RECOGNIZERS (unchanged logic, but digit
# patterns now also tolerate separators within digit spans)
# =========================================================

class AccountNumberRecognizer(EntityRecognizer):
    """
    Indian bank account numbers, 11–18 digits.
    Also matches digits written with spaces/dashes between them
    when preceded by an account-number keyword.
    Only the digit span is masked.
    """
    # Digit string with optional single separators between digits (11–18 digits)
    _DIG_SEP = r'\d(?:[\s.\-_]?\d){10,17}'

    _FULL = re.compile(
        r'(?i)(?:'
            r'account\s*(?:no\.?|number|#)?'
            r'|a\s*[/\-]?\s*c\s*(?:no\.?)?'
            r'|acct\.?'
            r'|acc\b\.?'
        r')\s*[:\-#]?\s*'
        r'(' + _DIG_SEP + r')',
    )

    def __init__(self):
        super().__init__(
            supported_entities=["ACCOUNT_NUMBER"],
            name="AccountNumberRecognizer",
        )

    def load(self):
        pass

    def analyze(self, text, entities, nlp_artifacts=None):
        results = []
        for m in self._FULL.finditer(text):
            digit_text = m.group(1)
            if not _valid_account(digit_text):
                continue
            results.append(RecognizerResult(
                entity_type="ACCOUNT_NUMBER",
                start=m.start(1),
                end=m.end(1),
                score=0.88,
            ))
        return results


class UANRecognizer(EntityRecognizer):
    """
    EPFO Universal Account Number — 12 digits.
    Also matches digits with separators when preceded by a UAN keyword.
    Only the digit span is masked.
    """
    _DIG_SEP = r'\d(?:[\s.\-_]?\d){11}'

    _FULL = re.compile(
        r'(?i)(?:'
            r'uan'
            r'|universal\s+account\s*(?:no\.?|number)?'
            r'|epfo\s*(?:no\.?|number)?'
            r'|pf\s+(?:uan|account)\s*(?:no\.?|number)?'
        r')\s*[:\-]?\s*'
        r'(' + _DIG_SEP + r')',
    )

    def __init__(self):
        super().__init__(
            supported_entities=["UAN"],
            name="UANRecognizer",
        )

    def load(self):
        pass

    def analyze(self, text, entities, nlp_artifacts=None):
        results = []
        for m in self._FULL.finditer(text):
            digit_text = m.group(1)
            if not _valid_uan(digit_text):
                continue
            results.append(RecognizerResult(
                entity_type="UAN",
                start=m.start(1),
                end=m.end(1),
                score=0.92,
            ))
        return results


# =========================================================
# OVERLAP RESOLUTION
# =========================================================

def _resolve_overlaps(results: list) -> list:
    """Keep the best (highest score, then longest span) non-overlapping results."""
    sorted_r = sorted(results, key=lambda r: (r.score, r.end - r.start), reverse=True)
    kept: list[RecognizerResult] = []
    for candidate in sorted_r:
        if not any(
            not (candidate.end <= k.start or candidate.start >= k.end)
            for k in kept
        ):
            kept.append(candidate)
    return kept


# =========================================================
# WHITESPACE REPAIR
# =========================================================

def _repair_whitespace(text: str) -> str:
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'(\w)\[', r'\1 [', text)
    text = re.sub(r'\]\s+([,.:;!?])', r']\1', text)
    return text


# =========================================================
# ENGINE SETUP
# =========================================================

def _build_analyzer() -> AnalyzerEngine:
    engine = None
    for model in ("en_core_web_lg", "en_core_web_md", "en_core_web_sm"):
        try:
            import spacy
            spacy.load(model)
            provider = NlpEngineProvider(nlp_configuration={
                "nlp_engine_name": "spacy",
                "models": [{"lang_code": "en", "model_name": model}],
            })
            engine = AnalyzerEngine(nlp_engine=provider.create_engine())
            print(f"[masking] spaCy model loaded: {model}")
            break
        except Exception:
            pass

    if engine is None:
        print("[masking] No spaCy model — regex-only mode.")
        engine = AnalyzerEngine()

    for cls in (
        CustomEmailRecognizer,
        AadhaarRecognizer,
        PANRecognizer,
        GSTRecognizer,
        IFSCRecognizer,
        IndianPhoneRecognizer,
        AccountNumberRecognizer,
        VoterIdRecognizer,
        PassportRecognizer,
        DrivingLicenceRecognizer,
        # UdyamRecognizer,
        UANRecognizer,
    ):
        engine.registry.add_recognizer(cls())

    return engine


_analyzer   = _build_analyzer()
_anonymizer = AnonymizerEngine()

_ENTITIES = [
    "EMAIL_ADDRESS",
    "AADHAAR", "PAN", "GST", "IFSC",
    "PHONE", "ACCOUNT_NUMBER",
    "VOTER_ID", "PASSPORT", "DL",
    # "UDYAM",
    "UAN",
]

_OPERATORS = {
    "EMAIL_ADDRESS":  OperatorConfig("replace", {"new_value": "[EMAIL]"}),
    "AADHAAR":        OperatorConfig("replace", {"new_value": "[AADHAAR]"}),
    "PAN":            OperatorConfig("replace", {"new_value": "[PAN]"}),
    "GST":            OperatorConfig("replace", {"new_value": "[GST]"}),
    "IFSC":           OperatorConfig("replace", {"new_value": "[IFSC]"}),
    "PHONE":          OperatorConfig("replace", {"new_value": "[PHONE]"}),
    "ACCOUNT_NUMBER": OperatorConfig("replace", {"new_value": "[ACCOUNT]"}),
    "VOTER_ID":       OperatorConfig("replace", {"new_value": "[VOTER_ID]"}),
    "PASSPORT":       OperatorConfig("replace", {"new_value": "[PASSPORT]"}),
    "DL":             OperatorConfig("replace", {"new_value": "[DL]"}),
    # "UDYAM":          OperatorConfig("replace", {"new_value": "[UDYAM]"}),
    "UAN":            OperatorConfig("replace", {"new_value": "[UAN]"}),
}


# =========================================================
# RESOURCE MONITOR
# =========================================================

class ResourceMonitor:
    """
    Context manager that measures CPU time, wall time, memory delta,
    peak memory, and thread count for any block of code.
    """

    def __init__(self, label: str = "block", print_report: bool = True):
        self.label        = label
        self.print_report = print_report
        self.stats: dict  = {}
        self._proc        = psutil.Process(os.getpid())

    def __enter__(self):
        mem_info          = self._proc.memory_info()
        self._wall_start  = time.perf_counter()
        self._cpu_start   = self._proc.cpu_times()
        self._mem_start   = mem_info.rss
        return self

    def __exit__(self, *_):
        wall_end  = time.perf_counter()
        cpu_end   = self._proc.cpu_times()
        mem_info  = self._proc.memory_info()

        wall_elapsed  = wall_end - self._wall_start
        cpu_user      = cpu_end.user  - self._cpu_start.user
        cpu_sys       = cpu_end.system - self._cpu_start.system
        mem_current   = mem_info.rss
        mem_delta     = mem_current - self._mem_start
        threads       = self._proc.num_threads()

        self.stats = {
            "label":            self.label,
            "wall_time_s":      round(wall_elapsed,  4),
            "cpu_user_s":       round(cpu_user,      4),
            "cpu_sys_s":        round(cpu_sys,       4),
            "mem_rss_mb":       round(mem_current  / 1024 / 1024, 2),
            "mem_delta_mb":     round(mem_delta     / 1024 / 1024, 2),
            "threads":          threads,
        }

        if self.print_report:
            self._print()

    def _print(self):
        s = self.stats
        print(
            f"\n{'─' * 50}\n"
            f"  Resource usage — {s['label']}\n"
            f"{'─' * 50}\n"
            f"  Wall time      : {s['wall_time_s']:.4f} s\n"
            f"  CPU user       : {s['cpu_user_s']:.4f} s\n"
            f"  CPU system     : {s['cpu_sys_s']:.4f} s\n"
            f"  Memory (RSS)   : {s['mem_rss_mb']:.2f} MB\n"
            f"  Memory delta   : {s['mem_delta_mb']:+.2f} MB\n"
            f"  Threads        : {s['threads']}\n"
            f"{'─' * 50}"
        )


# =========================================================
# PUBLIC API
# =========================================================

def mask_pii(text: str, monitor: bool = False) -> str:
    """
    Detect and replace Indian PII in *text*. Returns the anonymised string.

    Pipeline:
      1. _normalize_dense_separators  — collapse "char-sep-char-sep" runs
         so the main recognizers see compact tokens.
      2. Presidio analyze on the (possibly normalized) text.
      3. Overlap resolution.
      4. Presidio anonymize.
      5. Whitespace repair.

    Args:
        text:    Input string to mask.
        monitor: If True, prints a resource-usage report after each call.
    """
    with ResourceMonitor("mask_pii", print_report=monitor) as _mon:
        normalised = _normalize_dense_separators(text)
        raw        = _analyzer.analyze(text=normalised, language="en", entities=_ENTITIES)
        clean      = _resolve_overlaps(raw)
        result     = _anonymizer.anonymize(
                         text=normalised,
                         analyzer_results=clean,
                         operators=_OPERATORS,
                     )
        masked     = _repair_whitespace(result.text)
    return masked


# =========================================================
# TESTS
# =========================================================

if __name__ == "__main__":

    tests = [

        ("TEST 1 — Original bare values (no label)", """
        ECPPG0111K
        29ABCDE1234F1Z5
        HDFC0001234
        [Aadhaar Redacted]
        XGN3002623
        A2345671
        MH27 2012 0034761
        # UDYAM-DL-04-0012345
        9876543210
        user@example.com
        """),

        ("TEST 2 — With label prefix", """
        PAN no: ECPPG0111K
        GSTIN: 29ABCDE1234F1Z5
        IFSC HDFC0001234
        Aadhaar: [Aadhaar Redacted]
        Voter ID XGN3002623
        Passport A2345671
        DL MH27 2012 0034761
        # UDYAM-DL-04-0012345
        Mobile: 9876543210
        Email: user@example.com
        """),

        ("TEST 3 — Account / UAN context-anchored", """
        account no: 55678901234567
        UAN: 100234567890
        55678901234567
        100234567890
        """),

        ("TEST 4 — Mixed prose", """
        Hi, my Aadhaar is [Aadhaar Redacted] and my phone is 9 8 7 6 5 4 3 2 1 0.
        """),

        # ── NEW AMBIGUOUS-FORMAT TESTS ──────────────────────────────────

        ("TEST 5 — Space after every character", """
        PAN:  E C P P G 0 1 1 1 K
        IFSC: H D F C 0 0 0 1 2 3 4
        Voter ID: X G N 3 0 0 2 6 2 3
        Passport: A 2 3 4 5 6 7 1
        """),

        ("TEST 6 — Dash after every character", """
        PAN:  E-C-P-P-G-0-1-1-1-K
        GSTIN: 2-9-A-B-C-D-E-1-2-3-4-F-1-Z-5
        IFSC: H-D-F-C-0-0-0-1-2-3-4
        Aadhaar: [Aadhaar Redacted]
        Phone: 9-8-7-6-5-4-3-2-1-0
        """),

        ("TEST 7 — Random groupings / mixed separators", """
        PAN: ECPP G0 111K
        IFSC: HD FC00 01234
        Aadhaar: [Aadhaar Redacted]
        DL: MH-27 2012-003 4761
        # UDYAM: UDYAM DL 04 0012345
        account no: 5 5 6 7 8 9 0 1 2 3 4 5 6 7
        UAN: 1 0 0 2 3 4 5 6 7 8 9 0
        """),

        ("TEST 8 — Dot and underscore separators", """
        E.C.P.P.G.0.1.1.1.K
        IFSC: H_D_F_C_0_0_0_1_2_3_4
        Phone: 9.8.7.6.5.4.3.2.1.0
        Voter ID: X.G.N.3.0.0.2.6.2.3
        """),

        ("TEST 9 — In flowing prose with ambiguous IDs", """
        Respected Madam, I 2389 4539 1048 aged 145, Saket, Mumbai, Jharkhand, 862994 years residing at NEW PRESS Industries have lost my Udyam certificate. My registration number is OR-93-JB-1456 Aadhaar 132, Park Street, Ward 5, Nashik, Uttar Pradesh, 788787 PAN CWZ2786687. I need a duplicate certificate for bank loan application. My phone is Nikhil Joshi and email is 8880375323.
        """),

    ]

    for label, text in tests:
        print("\n" + "=" * 60)
        print(label)
        print("=" * 60)
        print("INPUT:\n", text)
        print("OUTPUT:\n", mask_pii(text, monitor=False))

# indian_pii_text_masker(1).py


In [ ]:
"""
Indian Text PII Masker (GLiNER + Presidio)
==========================================
Extracts and masks unstructured text-based entities using
GLiNER integrated into Microsoft Presidio.

Entities masked:
  PERSON        Person names
  ORGANIZATION  Company/org names
  ADDRESS       Postal addresses and landmarks
"""

import re
from typing import List, Dict, Optional

import torch

from presidio_analyzer import AnalyzerEngine, RecognizerResult, EntityRecognizer
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
from gliner import GLiNER


# =========================================================
# STRUCTURED IDENTIFIER PROTECTION
# =========================================================
# These patterns are replaced with sentinels BEFORE GLiNER runs,
# then restored verbatim AFTER masking.

_PROTECT_PATTERNS = [
    # AADHAAR — 12 digits in 4-4-4 groups (space, hyphen, or none)
    r"(?<!\d)\d{4}[\s\-]?\d{4}[\s\-]?\d{4}(?!\d)",
    # PAN — AAAAA9999A
    r"(?<![A-Z0-9])[A-Z]{5}\d{4}[A-Z](?![A-Z0-9])",
    # GST — 29ABCDE1234F1Z5
    r"\b\d{2}[A-Z]{5}\d{4}[A-Z]\d[Z][A-Z0-9]\b",
    # IFSC — 4 letters + 0 + 6 alphanumeric
    r"\b[A-Z]{4}0[A-Z0-9]{6}\b",
    # PHONE — Indian mobile numbers
    r"(?<!\d)(?:\+91[\s\-]?|91[\s\-]?|0)?[6-9]\d{4}[\s\-]?\d{5}(?!\d)",
    # ACCOUNT — context-anchored bank account
    r"(?<=account[\s:])\s*\d{11,18}(?!\d)",
    # VOTER_ID — 3 letters + 7 digits
    r"(?<![A-Z0-9])[A-Z]{3}\d{7}(?![A-Z0-9])",
    # PASSPORT — 1 letter + 7 digits
    r"(?<![A-Z0-9])[A-Z]\d{7}(?![A-Z0-9])",
    # DRIVING LICENCE — SS-RR-YYYY-NNNNNNN
    r"\b[A-Z]{2}[- ]\d{2}[- ]\d{4}[- ]\d{7}\b",
    # UDYAM number
    r"\b(?:UDYAM|UDAYM|UDHYAM)[- ][A-Z]{2}[- ]\d{2}[- ]\d{7}(?:/[A-Z]/\d{5})?",
    # UAM/URC numbers (alphanumeric codes like BR26D0018623, KR01A0000045)
    r"\b[A-Z]{2}\d{2}[A-Z]\d{7}\b",
    # UAN — context-anchored 12 digits
    r"(?<=uan[\s:])\s*\d{12}(?!\d)",
    # EMAIL
    r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}",
    # URLs
    r"https?://\S+",
    # ── Non-PII keywords GLiNER commonly mislabels as PERSON/ORG ──
    # Aadhaar/aadhar word variants
    r"\b(?:aadhaar|aadhar|adhar|addhar|e-aasdhr)"
    r"(?:\s+(?:number|no\.?|card|link|update|correction|enrollment|enrolment|seeding|linked|registered|validation|verified))?\b",
    # Udyam / Udyog / Udhyog keyword phrases
    r"\b(?:"
    r"udyam|udhyam|udaym|udayam"
    r"|udyam\s+registration|udhyam\s+registration|udaym\s+registration"
    r"|udyam\s+portal|udyam\s+number|udyam\s+no|udyam\s+certificate"
    r"|udyam\s+assist(?:\s+platform)?"
    r"|ud(?:y|h?y)og\s+a[ad]ha?ar"
    r"|ud(?:y|h?y)og\s+a[ad]ha?ar\s+(?:number|no\.?|registration|portal|certificate|memorandum|uam)"
    r"|udhyog\s+aadhaar\s+memorandum|udyog\s+aadhar\s+memorandum"
    r"|uam|urc"
    r")\b",
    # Registration / email / OTP / action words
    r"\b(?:registration|registartion|registrations)\b",
    r"\b(?:email|e-mail|email\s+id|mail\s+id)\b",
    r"\b(?:otp|otp\s+number)\b",
    r"\b(?:cancell?(?:ation)?|cancel(?:l?ed)?)\b",
    r"\b(?:solve|solved|solution|resolve|resolved)\b",
    r"\b(?:alr?ea?dy|alrady)\b",
    r"\b(?:unable|trying|clicking|applying|attempting|writing|didt)\b",
    r"\b(?:sir|madam|mam|dear\s+sir|dear\s+madam|dear\s+sir\s*/\s*madam)\b",
    r"\b(?:proprietor|sole\s+proprietor|respondent|applicant|complainant)\b",
    r"\b(?:application\s+)?dt\.?\b",
    r"\b(?:latitude|longitude|geolocation|geo\s*tag(?:ging)?)\b",
    r"\b(?:letter\s*head|letterhead)\b",
    r"\b(?:msme|msefc|msmed|pmegp|kvic|sidbi|nsic|cgtmse)\b",
]

_PROTECT_RE = re.compile(
    "|".join(f"(?:{p})" for p in _PROTECT_PATTERNS),
    re.IGNORECASE,
)

_SENTINEL_PREFIX = "__PROTECT_"
_SENTINEL_SUFFIX = "__"


def _protect_identifiers(text: str) -> tuple[str, Dict[str, str]]:
    restore_map: Dict[str, str] = {}
    counter = [0]

    def _repl(m):
        key = f"{_SENTINEL_PREFIX}{counter[0]}{_SENTINEL_SUFFIX}"
        restore_map[key] = m.group(0)
        counter[0] += 1
        return key

    return _PROTECT_RE.sub(_repl, text), restore_map


def _restore_identifiers(text: str, restore_map: Dict[str, str]) -> str:
    for sentinel, original in restore_map.items():
        text = text.replace(sentinel, original)
    return text


# =========================================================
# PERSON SPAN VALIDATOR
# =========================================================
# Post-GLiNER guard: rejects PERSON predictions that are clearly not names.

# Business/generic nouns that end up in 2-token "X Business" spans
_BUSINESS_TAIL_WORDS = {
    "business", "enterprise", "firm", "company", "shop", "store",
    "agency", "bureau", "centre", "center", "services", "service",
    "trading", "industries", "industry", "corporation", "associates",
    "ventures", "venture", "works", "solutions", "consultancy",
    "suppliers", "supplier", "dealers", "dealer", "products",
    "exports", "imports", "logistics", "technologies", "tech",
    "systems", "system", "group", "holding", "holdings",
}

# Single tokens that look superficially name-like but aren't
_SINGLE_TOKEN_NON_NAMES = {
    # Misspellings / typos / action words
    "claear", "didt", "canu", "forgut", "alrady", "cancell",
    "issie", "privde", "provied", "provied", "recev", "resev",
    "resubmit", "migrate", "retrieve", "download", "upload",
    # Document/portal words
    "attachment", "letterhead", "certificate", "document", "letter",
    "annexure", "enclosure", "affidavit", "invoice", "receipt",
    "bharatmapservice", "webgis", "portal", "website",
    # Commodities / generic nouns
    "milk", "cement", "rice", "wheat", "cotton", "gold", "silver",
    "iron", "steel", "wood", "cloth", "cloth",
    # Roles already in protect list but belt-and-suspenders
    "respondent", "petitioner", "complainant", "proprietor",
    "applicant", "director", "manager", "owner", "partner",
}

# Name honorifics / prefixes — single-token spans with these are NOT names
_HONORIFICS = {"mr", "mrs", "ms", "dr", "shri", "smt", "prof", "er"}

# Name particles allowed as single lowercase tokens
_NAME_PARTICLES = {"ji", "kumar", "devi", "bai", "lal", "ram", "singh",
                   "devi", "ben", "bhai", "rao", "nair", "das"}


def _is_valid_person_span(span_text: str) -> bool:
    """Return False if the span is clearly not a person name."""
    raw = span_text.strip()
    if not raw:
        return False

    # Contains a sentinel — GLiNER tagged a protected token
    if "__PROTECT_" in raw:
        return False

    # Pure digits / punctuation
    if re.fullmatch(r"[\d\s\.\-/,;:]+", raw):
        return False

    # Strip trailing punctuation artifacts (e.g. "Rishi/s" → "Rishi")
    cleaned = re.sub(r"[/\\.,;:!?\s]+$", "", raw).strip()
    if not cleaned:
        return False

    norm = cleaned.lower()
    tokens = norm.split()

    # Single token checks
    if len(tokens) == 1:
        t = tokens[0]
        # All-lowercase single token that's not a known particle
        if raw == raw.lower() and t not in _NAME_PARTICLES:
            return False
        # Known non-name single token
        if t in _SINGLE_TOKEN_NON_NAMES:
            return False
        # Honorific alone (no actual name)
        if t.rstrip(".") in _HONORIFICS:
            return False
        # All-uppercase token that's 6+ chars and contains no vowels
        # (likely an acronym or garbled word, e.g. CLAEAR, DIDT)
        if raw.isupper() and len(t) >= 4:
            vowels = set("aeiou")
            if not any(c in vowels for c in t):
                return False

    # Multi-token: last token is a generic business noun → ORG, not NAME
    if len(tokens) >= 2 and tokens[-1] in _BUSINESS_TAIL_WORDS:
        return False

    # Multi-token: ALL tokens are lowercase and none are name particles
    if all(tok == tok.lower() for tok in tokens):
        if not any(tok in _NAME_PARTICLES for tok in tokens):
            return False

    # Span is longer than 6 tokens — very unlikely to be a single person name
    if len(tokens) > 6:
        return False

    return True


# =========================================================
# POST-MASK CLEANUP
# =========================================================

_BROKEN_SENTINEL_RE = re.compile(
    r"(?:\[)?__(?:PROTECT_\d+|\[(?:NAME|ORG|ADDRESS)\])__(?:\])?",
    re.IGNORECASE,
)
_DOUBLE_BRACKET_RE = re.compile(r"\[\[([A-Z]+)\]\]")


def _cleanup_artefacts(text: str) -> str:
    text = _BROKEN_SENTINEL_RE.sub("", text)
    text = _DOUBLE_BRACKET_RE.sub(r"[\1]", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()


# =========================================================
# GLINER RECOGNIZER
# =========================================================

_ENTITY_THRESHOLDS: Dict[str, float] = {
    "PERSON":       0.75,
    "ORGANIZATION": 0.85,
    "ADDRESS":      0.80,
}

_ADDRESS_MIN_TOKENS: int = 2


class GlinerRecognizer(EntityRecognizer):
    FINETUNED_LABEL_MAPPING = {
        "full_name":      "PERSON",
        "company_name":   "ORGANIZATION",
        "postal_address": "ADDRESS",
    }

    def __init__(self, finetuned_model_path: str = "./gliner_pii_finetuned"):
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

        print(f"[text-masking] Loading GLiNER from '{finetuned_model_path}' on {device.upper()} ...")
        self.finetuned_model = GLiNER.from_pretrained(finetuned_model_path).to(device)
        self.finetuned_labels = list(self.FINETUNED_LABEL_MAPPING.keys())
        print("[text-masking] GLiNER model loaded and ready.")

        super().__init__(
            supported_entities=list(self.FINETUNED_LABEL_MAPPING.values()),
            name="GlinerRecognizer",
        )

    def load(self):
        pass

    def analyze(self, text: str, entities: List[str], nlp_artifacts=None) -> List[RecognizerResult]:
        results = []
        gliner_floor = min(_ENTITY_THRESHOLDS.values())

        for pred in self.finetuned_model.predict_entities(
            text, self.finetuned_labels, threshold=gliner_floor
        ):
            presidio_entity = self.FINETUNED_LABEL_MAPPING.get(pred["label"])
            if not presidio_entity or presidio_entity not in entities:
                continue
            if pred["score"] < _ENTITY_THRESHOLDS[presidio_entity]:
                continue

            span_text = pred["text"]

            # Reject if span overlaps a sentinel
            if "__PROTECT_" in span_text:
                continue

            # Entity-specific validation
            if presidio_entity == "PERSON":
                if not _is_valid_person_span(span_text):
                    continue

            elif presidio_entity == "ADDRESS":
                if len(span_text.split()) < _ADDRESS_MIN_TOKENS:
                    continue

            results.append(RecognizerResult(
                entity_type=presidio_entity,
                start=pred["start"],
                end=pred["end"],
                score=pred["score"],
            ))

        return _dedup_results(results)


def _dedup_results(results: List[RecognizerResult]) -> List[RecognizerResult]:
    ranked = sorted(results, key=lambda r: (r.score, r.end - r.start), reverse=True)
    kept: List[RecognizerResult] = []
    for r in ranked:
        if not any(r.start < k.end and r.end > k.start for k in kept):
            kept.append(r)
    return kept


# =========================================================
# ENGINE SETUP
# =========================================================

_FINETUNED_PATH = "C:/College/PS-1/PII Masking/gliner_pii_finetuned"

_analyzer = AnalyzerEngine()
_analyzer.registry.add_recognizer(GlinerRecognizer(finetuned_model_path=_FINETUNED_PATH))
_anonymizer = AnonymizerEngine()

_ENTITIES = ["PERSON", "ORGANIZATION", "ADDRESS"]

_OPERATORS = {
    "PERSON":       OperatorConfig("replace", {"new_value": "[NAME]"}),
    "ORGANIZATION": OperatorConfig("replace", {"new_value": "[ORG]"}),
    "ADDRESS":      OperatorConfig("replace", {"new_value": "[ADDRESS]"}),
}


# =========================================================
# PUBLIC API
# =========================================================

def mask_text_entities(text: str) -> str:
    # Step 1: Protect structured identifiers and known non-PII keywords
    protected_text, restore_map = _protect_identifiers(text)

    # Step 2: Run GLiNER via Presidio
    hits = _analyzer.analyze(text=protected_text, language="en", entities=_ENTITIES)
    redacted = _anonymizer.anonymize(
        text=protected_text,
        analyzer_results=hits,
        operators=_OPERATORS,
    )

    # Step 3: Restore protected originals
    output = _restore_identifiers(redacted.text, restore_map)

    # Step 4: Clean up broken sentinel artefacts
    output = _cleanup_artefacts(output)
    return output


# =========================================================
# TESTING & EXAMPLES
# =========================================================

if __name__ == "__main__":
    print("Initializing PII Masker... (This may take a moment to load the models)")

    test_cases = [
        # Correct masking
        "My name is Rahul Sharma and my contact number is +91-9876543210. Please email me at rahul.sharma@gmail.com.",
        "The sole proprietor, Mr. Amit Kumar, applied for UDYAM registration. Udyam number: UDYAM-MH-18-0123456.",
        "I am Dr. Sneha Desai. I live at Flat No 402, Sunshine Tower, MG Road, Bangalore.",
        "Tata Consultancy Services is located in Pune. The Managing Director, Rajesh Gopinathan, signed the document.",
        # False positive cases from real data
        "DEAR SIR, I AM A PROPRIETOR HAVING PAN: GZDPD7433L AND AADHAR NUMBER: 9759 9062 6267.",
        "I am running a Cement Business and want to get registered with MSME.",
        "then I failed to enter OTP as my earlier mobile number has been changed.",
        "Udyam Registration (UDYAM-WB-05-0001331) Certificate.",
        "I want to update my phone number and email id in my UAM that is BR26D0018623.",
        "When i am trying to do registration after Aadhar validation successful.",
        "Application dt.06/06/2024 is still not converted in case.",
        "UDHYOG AADHAAR MEMORANDUM HAS ALREADY BEEN DONE THROUGH THIS AADHAR NUMBER.",
        "Udyog Aadhaar Number: MH18D0032045  Enterprise Name: P. O. P. Decorator  Date of Registration: 20/05/2018",
    ]

    print("\n" + "=" * 60)
    print("RUNNING PII MASKING EXAMPLES")
    print("=" * 60 + "\n")

    for i, text in enumerate(test_cases, 1):
        print(f"--- Example {i} ---")
        print(f"ORIGINAL: {text}")
        masked = mask_text_entities(text)
        print(f"MASKED:   {masked}\n")

    print("Testing complete.")

# api.py


In [ ]:
"""
Indian PII Masker — REST API
============================
Wraps the Two-Pass PII Masking Pipeline in a FastAPI application.
Pass 1: Strict Regex (indian_pii_masker.py)
Pass 2: Contextual NLP (indian_gliner_pii_masker.py)

Endpoints
─────────
  POST /mask          Mask PII in a text string
  POST /mask/batch    Mask PII in multiple texts at once
  GET  /health        Liveness check + resource snapshot

Setup
─────
    pip install fastapi uvicorn psutil gliner presidio-analyzer presidio-anonymizer

Run
───
    uvicorn api:app --reload --port 8000

    # or directly:
    python api.py
"""

from __future__ import annotations

import os
import time

import psutil
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

# Import Pass 1 (Strict IDs) and the utility monitor
from indian_pii_masker import mask_pii, ResourceMonitor
# Import Pass 2 (Contextual NLP)
from indian_pii_text_masker import mask_text_entities


# =========================================================
# APP
# =========================================================

app = FastAPI(
    title="Indian PII Masker",
    description="Detects and masks Indian PII using a Two-Pass Pipeline (Regex -> GLiNER).",
    version="2.0.0",
)

_proc = psutil.Process(os.getpid())


# =========================================================
# MIDDLEWARE
# =========================================================

@app.middleware("http")
async def log_response_time(request: Request, call_next):
    """
    Middleware to calculate and print the total response time for every request.
    Also injects the timing into the response headers.
    """
    start_time = time.perf_counter()
    
    response = await call_next(request)
    
    process_time = time.perf_counter() - start_time
    print(f"[{request.method}] {request.url.path} - Total Response Time: {process_time:.4f} seconds")
    
    # Optional: Attach the process time to the response headers
    response.headers["X-Process-Time"] = str(process_time)
    
    return response


# =========================================================
# REQUEST / RESPONSE MODELS
# =========================================================

class MaskRequest(BaseModel):
    text: str = Field(..., description="Input text to mask.")
    monitor: bool = Field(False, description="Include resource usage stats in the response.")


class MaskResponse(BaseModel):
    masked_text: str
    resources: dict | None = None


class BatchMaskRequest(BaseModel):
    texts: list[str] = Field(..., description="List of input texts to mask.")
    monitor: bool = Field(False, description="Include per-item resource usage stats.")


class BatchMaskResponse(BaseModel):
    results: list[MaskResponse]


class HealthResponse(BaseModel):
    status: str
    uptime_s: float
    memory_rss_mb: float
    cpu_percent: float
    threads: int


# =========================================================
# STARTUP
# =========================================================

_start_time = time.perf_counter()


# =========================================================
# ENDPOINTS
# =========================================================

@app.post(
    "/mask",
    response_model=MaskResponse,
    summary="Mask PII in a single text using a two-pass pipeline",
)
def mask_single(req: MaskRequest) -> MaskResponse:
    """
    Executes a Two-Pass Redaction:
    1. Evaluates strict structured IDs via regex.
    2. Passes the redacted string to GLiNER to capture context-heavy names and addresses.
    """
    if not req.text.strip():
        raise HTTPException(status_code=422, detail="'text' must not be empty.")

    # Wrap the entire two-step process in one monitor block to get cumulative latency/CPU metrics
    with ResourceMonitor("two_pass_masking", print_report=False) as mon:
        step1_masked = mask_pii(req.text, monitor=False)
        final_masked = mask_text_entities(step1_masked)

    return MaskResponse(
        masked_text=final_masked,
        resources=mon.stats if req.monitor else None,
    )


@app.post(
    "/mask/batch",
    response_model=BatchMaskResponse,
    summary="Mask PII in multiple texts",
)
def mask_batch(req: BatchMaskRequest) -> BatchMaskResponse:
    """
    Accepts a list of text strings and returns each one masked.
    Routes each string through both the strict ID and NLP masking modules.
    """
    if not req.texts:
        raise HTTPException(status_code=422, detail="'texts' list must not be empty.")

    results = []
    for text in req.texts:
        with ResourceMonitor("two_pass_batch_item", print_report=False) as mon:
            step1_masked = mask_pii(text, monitor=False)
            final_masked = mask_text_entities(step1_masked)
            
        results.append(MaskResponse(
            masked_text=final_masked,
            resources=mon.stats if req.monitor else None,
        ))

    return BatchMaskResponse(results=results)


@app.get(
    "/health",
    response_model=HealthResponse,
    summary="Liveness check",
)
def health() -> HealthResponse:
    """Returns server status and a current resource snapshot."""
    mem   = _proc.memory_info()
    cpu   = _proc.cpu_percent(interval=0.1)
    return HealthResponse(
        status="ok",
        uptime_s=round(time.perf_counter() - _start_time, 2),
        memory_rss_mb=round(mem.rss / 1024 / 1024, 2),
        cpu_percent=cpu,
        threads=_proc.num_threads(),
    )


# =========================================================
# ENTRYPOINT
# =========================================================

if __name__ == "__main__":
    uvicorn.run("api:app", host="0.0.0.0", port=8000, reload=True)

# run_pii_masker.py


In [ ]:
"""
Batch PII Masker Runner
========================
Reads english_queries.txt, sends all queries to the /mask/batch endpoint,
and saves the masked results to masked_output.txt and masked_output.csv.

Usage
-----
    python run_pii_masker.py

    # Custom file or API URL:
    python run_pii_masker.py --input english_queries.txt --url http://localhost:8000

Requirements
------------
    pip install requests
"""

import argparse
import csv
import json
import sys
from pathlib import Path

import requests


# ── Config ────────────────────────────────────────────────────────────────────

DEFAULT_INPUT = "english_queries.txt"
DEFAULT_URL   = "http://localhost:8000"
TXT_OUTPUT    = "masked_output.txt"
CSV_OUTPUT    = "masked_output.csv"


# ── Parse queries from file ───────────────────────────────────────────────────

def parse_queries(filepath: str) -> list[dict]:
    """
    Parses the text file into a list of {grievance_no, text} dicts.
    Each entry is separated by a blank line and formatted as:
        GRIEVANCENO: description text
    """
    content = Path(filepath).read_text(encoding="utf-8")
    blocks  = [b.strip() for b in content.split("\n\n") if b.strip()]

    queries = []
    for block in blocks:
        if ": " in block:
            grievance_no, _, text = block.partition(": ")
            queries.append({"grievance_no": grievance_no.strip(), "text": text.strip()})
        else:
            print(f"  [WARN] Skipping unrecognised block: {block[:60]}...")

    return queries


# ── Call API ──────────────────────────────────────────────────────────────────

def call_batch_api(texts: list[str], base_url: str) -> list[str]:
    url     = f"{base_url.rstrip('/')}/mask/batch"
    payload = {"texts": texts, "monitor": False}

    print(f"  Sending {len(texts)} queries to {url} ...")
    try:
        resp = requests.post(url, json=payload, timeout=400)
        resp.raise_for_status()
    except requests.exceptions.ConnectionError:
        print(f"\n[ERROR] Could not connect to {url}")
        print("  Make sure your API is running:  uvicorn api:app --reload --port 8000")
        sys.exit(1)
    except requests.exceptions.HTTPError as e:
        print(f"\n[ERROR] API returned an error: {e}")
        print(f"  Response: {resp.text[:500]}")
        sys.exit(1)

    data = resp.json()
    return [item["masked_text"] for item in data["results"]]


# ── Save outputs ──────────────────────────────────────────────────────────────

def save_txt(queries: list[dict], masked_texts: list[str], filepath: str):
    lines = []
    for q, masked in zip(queries, masked_texts):
        lines.append(f"{q['grievance_no']}: {masked}")
        lines.append("")          # blank line separator
    Path(filepath).write_text("\n".join(lines), encoding="utf-8")
    print(f"  Saved text output  → {filepath}")


def save_csv(queries: list[dict], masked_texts: list[str], filepath: str):
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["GrievanceNo", "OriginalText", "MaskedText"])
        for q, masked in zip(queries, masked_texts):
            writer.writerow([q["grievance_no"], q["text"], masked])
    print(f"  Saved CSV output   → {filepath}")


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Run all queries through the PII masker batch API.")
    parser.add_argument("--input", default=DEFAULT_INPUT, help="Path to english_queries.txt")
    parser.add_argument("--url",   default=DEFAULT_URL,   help="Base URL of the PII masker API")
    args = parser.parse_args()

    print(f"\n[1/4] Reading queries from: {args.input}")
    queries = parse_queries(args.input)
    print(f"      Found {len(queries)} queries.")

    print(f"\n[2/4] Calling /mask/batch ...")
    texts        = [q["text"] for q in queries]
    masked_texts = call_batch_api(texts, args.url)
    print(f"      Done. {len(masked_texts)} responses received.")

    print(f"\n[3/4] Saving outputs ...")
    save_txt(queries, masked_texts, TXT_OUTPUT)
    save_csv(queries, masked_texts, CSV_OUTPUT)

    print(f"\n[4/4] All done! ✓")
    print(f"      {len(queries)} queries masked successfully.\n")


if __name__ == "__main__":
    main()